# I.Data Ingestion

This notebook purpose is to turn the data in a format and structure usable for further analysis.
 - Each tool data is stored in a parquet by tool with a unique identifier by application for the tool in `Data/Output Data/data`.
 - the metadata for each tool and application is stored within `metadata.db`, one table per section of the raw files, all linked by `file_id` (+ `tool_ref`, `source_file`):

| Table | Rows | Content |
|---|---|---|
| `tools` | 1 per tool (10) | `tool_ref`, tool name, source folder |
| `general_infos` | 1 per file (730) | date, general info of the application |
| `control_box` | 1 per file | control box settings |
| `motor_box` | 1 per file | motor settings |
| `head` | 1 per file | drilling head info |
| `pset` | 1 per file | pset (program set) used |
| `cycle_parameters` | 1 per file | cycle limits, retract, tool breakage detection... |
| `results` | 1 per file | global result of the application |
| `program` | 1 per step (2728) | programmed parameters of each step |
| `step_results` | 1 per step (2285) | measured results of each step (duration, stop code, max amps...) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import StringIO


from pathlib import Path
from tqdm import tqdm
import os, time, sys
import sqlite3
import shutil

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "helper.py").exists()), Path.cwd())
NOTEBOOK_PATH = PROJECT_ROOT
DATA_PATH = NOTEBOOK_PATH / "Data/04 Qualification Outils Coupants"
OUTPUT_PATH = NOTEBOOK_PATH / "Data/Output Data"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"Data path {DATA_PATH} does not exist"
assert OUTPUT_PATH.exists(), f"Output path {OUTPUT_PATH} does not exist"

data_folder = os.listdir(DATA_PATH)
datafiles = {}

for folder in sorted(p for p in data_folder if (DATA_PATH / p).is_dir()):
    folder_path = DATA_PATH / folder
    datafiles[folder] = os.listdir(folder_path)

print(datafiles)

In [ ]:
# Section header as written in the file -> clean SQL table name
SECTIONS = {
    "*** General Infos ***": "general_infos",
    "*** Control Box Datas ***": "control_box",
    "*** Motor Datas ***": "motor_box",
    "*** Head Datas ***": "head",
    "*** Pset ***": "pset",
    "*** Cycle Parameters ***": "cycle_parameters",
    "*** Program ***": "program",
    "*** Results ***": "results",
    "*** Step Results ***": "step_results",
    "*** Datas ***": "data",
}
# Sections that are a real table (header + N rows); all others are header/value line pairs -> 1 row per file
TABLE_SECTIONS = {"program", "step_results", "data"}

DB_PATH = OUTPUT_PATH / "metadata.db"
PARQUET_PATH = OUTPUT_PATH / "data"   # one sub-folder per tool: data/tool_ref=DT00037/...

def read_csv(text):
    "File reader because the files are indeed UTF-8 not excel files"
    return pd.read_csv(StringIO(text), sep="\t", decimal=",", skipinitialspace=True)

def split_sections(file_path):
    """
    Split sections by returning a dict of pairs section name -> number of lines
    Output: {"header": ["l1", "l2"], "body": ["l3"], "footer": ["l4"]}
    """
    lines = [l.rstrip("\t\r") for l in Path(file_path).read_text(encoding="utf-8").split("\n")]
    starts = [(i, SECTIONS[l.strip()]) for i, l in enumerate(lines) if l.strip() in SECTIONS]
    starts.append((len(lines), None)) #end marker
    return {
        name: lines[start + 1:end]
        for (start, name), (end, _) in zip(starts, starts[1:])
    }

def parse_table(lines):
    "reads a section that is a real table (header + N-rows)"
    content = "\n".join(l for l in lines if l.strip())
    return read_csv(content) if content else None

def parse_pairs(lines):
    """
    reads a section that is a list of header/value pairs (1 row per file)
    returns a DataFrame with 1 row and N columns.

    Ex input: [
    "A\tB",     
    "1\t2",      
    "",          
    "Comment",   
    "",          
    "C\tD",      
    "3\t4",     
]
    """
    parts, i = [], 0
    while i < len(lines):
        if not lines[i].strip():
            i += 1
            continue
        header, values = lines[i], lines[i + 1] if i + 1 < len(lines) else ""
        if values.strip():
            parts.append(read_csv(header + "\n" + values))
        else:
            parts.append(pd.DataFrame({c.strip(): [None] for c in header.split("\t")}))
        i += 2
    return pd.concat(parts, axis=1) if parts else None

def parse_section(name, lines):
    "Parse using the appropriate parser for the section type (table vs header/value pairs)"
    df = parse_table(lines) if name in TABLE_SECTIONS else parse_pairs(lines)
    if df is None:
        return None
    df.columns = [str(c).strip() for c in df.columns]
    return df

metadata_frames = {name: [] for name in SECTIONS.values() if name != "data"}
data_frames = []
tools = []
errors = []

tool_folders = sorted(p for p in DATA_PATH.iterdir() if p.is_dir())
file_id = 0
for tool_folder in tqdm(tool_folders, desc="Processing tool folders"):
    tool_ref, _, tool_name = tool_folder.name.partition(" - ")  # "DT00037", "guhring bomb Ø11.1"
    tools.append({"tool_ref": tool_ref, "tool_name": tool_name, "folder": tool_folder.name})

    for f in sorted(p for p in tool_folder.iterdir() if p.suffix.lower() == ".xls"):
        try:
            #parser dict. Name -> dataframe
            parsed = {name: parse_section(name, lines) for name, lines in split_sections(f).items()}
        except Exception as e:
            errors.append((str(f.relative_to(DATA_PATH)), repr(e)))
            continue

        for name, df in parsed.items():
            #We add the unique identifiers to each dataframe
            if df is None:
                continue
            df.insert(0, "file_id", file_id)
            df.insert(1, "tool_ref", tool_ref)
            df.insert(2, "source_file", f.name)
            (data_frames if name == "data" else metadata_frames[name]).append(df)
        file_id += 1

print(f"{file_id} files parsed, {len(errors)} failed")
for path, err in errors:
    print("  FAILED:", path, "->", err)

# metadata -> SQLite (one table per section, linked by file_id)
tables = {name: pd.concat(frames, ignore_index=True) for name, frames in metadata_frames.items() if frames}
tables["tools"] = pd.DataFrame(tools)
tables["general_infos"]["Date"] = pd.to_datetime(tables["general_infos"]["Date"], format="%Y-%m-%d:%H:%M:%S")

DB_PATH.unlink(missing_ok=True)
conn = sqlite3.connect(DB_PATH)
for name, df in tables.items():
    df.to_sql(name, conn, index=False)
conn.close()
print("SQLite ->", DB_PATH)
for name, df in tables.items():
    print(f"  {name:<17} {df.shape[0]:>5} rows x {df.shape[1]} cols")

# measurement data / Parquet dataset partitioned by tool
data = pd.concat(data_frames, ignore_index=True)
data["tool_ref"] = data["tool_ref"].astype("category")
data["source_file"] = data["source_file"].astype("category")

# a column that is numeric in some files and text in others can't be stored by Parquet as-is
for col in data.columns[data.dtypes == object]:
    print(f"  column '{col}' has mixed types -> stored as string")
    data[col] = data[col].astype("string")
shutil.rmtree(PARQUET_PATH, ignore_errors=True)  # partitioned writes add files, so start clean
data.to_parquet(PARQUET_PATH, partition_cols=["tool_ref"], index=False)
parts = sorted(PARQUET_PATH.glob("tool_ref=*/*.parquet"))
print(f"Parquet -> {PARQUET_PATH}\n  {len(data):,} rows x {data.shape[1]} cols in {len(parts)} files, "
      f"{sum(p.stat().st_size for p in parts) / 1e6:.0f} MB")